# REALITY LENS AI

Interactive data investigation project. Run the single code cell below. It installs missing dependencies, creates synthetic demonstration data, generates visualizations and exports five CSV reports.

In [ ]:
# REALITY LENS AI — complete one-cell project
import sys, subprocess, importlib.util, warnings
warnings.filterwarnings("ignore")
for module, package in {"numpy":"numpy", "pandas":"pandas", "plotly":"plotly", "sklearn":"scikit-learn", "scipy":"scipy"}.items():
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])
import numpy as np, pandas as pd, plotly.express as px, plotly.graph_objects as go
from IPython.display import display, HTML
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from scipy.stats import kruskal
np.random.seed(42)
display(HTML('<div style="padding:24px;background:#172554;color:white;border-radius:12px"><h1>REALITY LENS AI</h1><p>Interactive data investigation and hidden pattern discovery</p></div>'))
# Synthetic dataset
n=2000
regions=np.random.choice(["North","South","East","West"],n)
education=np.random.choice(["School","Graduate","Postgraduate"],n,p=[.25,.50,.25])
age=np.random.randint(18,66,n)
rb={"North":8000,"South":12000,"East":5000,"West":15000}; eb={"School":0,"Graduate":10000,"Postgraduate":20000}
income=np.array([rb[r]+eb[e]+a*450 for r,e,a in zip(regions,education,age)])+np.random.normal(0,12000,n)
income=np.clip(income,5000,None)
spending=np.clip(income*np.random.uniform(.30,.80,n)+np.random.normal(0,5000,n),500,None)
savings=income-spending
satisfaction=np.clip(5+4*savings/np.maximum(income,1)+np.random.normal(0,1.5,n),1,10)
df=pd.DataFrame({"Customer_ID":range(1,n+1),"Age":age,"Region":regions,"Education":education,"Monthly_Income":income.round(2),"Monthly_Spending":spending.round(2),"Monthly_Savings":savings.round(2),"Satisfaction_Score":satisfaction.round(1)})
# Add controlled quality issues
mi=np.random.choice(df.index,100,replace=False); df.loc[mi[:40],"Monthly_Income"]=np.nan; df.loc[mi[40:70],"Education"]=np.nan; df.loc[mi[70:],"Satisfaction_Score"]=np.nan
ai=np.random.choice(df.index,25,replace=False); df.loc[ai,"Monthly_Spending"]*=np.random.uniform(3,7,len(ai))
df=pd.concat([df,df.sample(15,random_state=42)],ignore_index=True)
print(f"DATASET: {len(df):,} rows x {len(df.columns)} columns | Missing cells: {df.isna().sum().sum()} | Duplicate rows: {df.duplicated().sum()}")
display(df.head(10))
# Data profile
profile=pd.DataFrame({"Data_Type":df.dtypes.astype(str),"Missing_Count":df.isna().sum(),"Missing_Percentage":(df.isna().mean()*100).round(2),"Unique_Values":df.nunique()})
print("DATA QUALITY PROFILE"); display(profile)
mp=profile["Missing_Percentage"].rename_axis("Feature").reset_index()
px.bar(mp,x="Missing_Percentage",y="Feature",orientation="h",color="Missing_Percentage",title="Missing Data Investigation",template="plotly_white").show()
numeric=["Age","Monthly_Income","Monthly_Spending","Monthly_Savings","Satisfaction_Score"]
print("DESCRIPTIVE STATISTICS"); display(df[numeric].describe().T.round(2))
for col in numeric: px.histogram(df,x=col,nbins=35,marginal="box",title=f"Distribution and Outliers: {col}",template="plotly_white").show()
px.imshow(df[numeric].corr(),text_auto=".2f",aspect="auto",color_continuous_scale="RdBu_r",zmin=-1,zmax=1,title="Feature Correlation Matrix",template="plotly_white").show()
# Regional comparison
region_summary=df.groupby("Region",observed=True).agg(Average_Income=("Monthly_Income","mean"),Median_Income=("Monthly_Income","median"),Average_Spending=("Monthly_Spending","mean"),Median_Spending=("Monthly_Spending","median"),Average_Savings=("Monthly_Savings","mean"),Customer_Count=("Customer_ID","count")).round(2)
print("REGIONAL COMPARISON"); display(region_summary)
px.bar(region_summary.reset_index(),x="Region",y=["Average_Income","Average_Spending","Average_Savings"],barmode="group",title="Regional Financial Comparison",template="plotly_white").show()
mm=df.groupby("Region",observed=True)["Monthly_Spending"].agg(["mean","median","std","count"]).round(2); print("MEAN VS MEDIAN"); display(mm)
fig=go.Figure(); fig.add_bar(x=mm.index,y=mm["mean"],name="Mean"); fig.add_bar(x=mm.index,y=mm["median"],name="Median"); fig.update_layout(title="Mean vs Median Spending by Region",barmode="group",template="plotly_white"); fig.show()
# IQR outliers
out=[]
for col in numeric:
    v=df[col].dropna(); q1,q3=v.quantile([.25,.75]); iqr=q3-q1; lo,hi=q1-1.5*iqr,q3+1.5*iqr
    out.append({"Feature":col,"Q1":round(q1,2),"Q3":round(q3,2),"IQR":round(iqr,2),"Lower_Bound":round(lo,2),"Upper_Bound":round(hi,2),"Potential_Outliers":int(((v<lo)|(v>hi)).sum())})
outlier_table=pd.DataFrame(out); print("IQR OUTLIER REPORT"); display(outlier_table)
px.box(df,x="Region",y="Monthly_Spending",color="Region",title="Spending Distribution and Potential Outliers",template="plotly_white").show()
px.scatter(df.dropna(subset=["Monthly_Income","Monthly_Spending"]),x="Monthly_Income",y="Monthly_Spending",color="Region",hover_data=["Age","Education"],title="Income vs Spending",opacity=.7,template="plotly_white").show()
# ML anomaly detection
ml=df[numeric].replace([np.inf,-np.inf],np.nan); ml=ml.fillna(ml.median()); scaled=StandardScaler().fit_transform(ml)
model=IsolationForest(n_estimators=200,contamination=.025,random_state=42); pred=model.fit_predict(scaled)
df["Anomaly_Status"]=np.where(pred==-1,"Potential Anomaly","Typical"); df["Anomaly_Score"]=-model.score_samples(scaled)
print("POTENTIAL ANOMALIES:",int((pred==-1).sum()),"(statistical flags, not proof of fraud/error)")
display(df[df.Anomaly_Status=="Potential Anomaly"][["Customer_ID","Region","Monthly_Income","Monthly_Spending","Monthly_Savings","Anomaly_Score"]].sort_values("Anomaly_Score",ascending=False).head(15))
px.scatter(df,x="Monthly_Income",y="Monthly_Spending",color="Anomaly_Status",hover_data=["Customer_ID","Region"],title="Isolation Forest Anomaly Detection",template="plotly_white").show()
# PCA
pca=PCA(n_components=2); pv=pca.fit_transform(scaled); pca_df=pd.DataFrame({"PC1":pv[:,0],"PC2":pv[:,1],"Anomaly_Status":df["Anomaly_Status"],"Region":df["Region"]})
px.scatter(pca_df,x="PC1",y="PC2",color="Anomaly_Status",hover_data=["Region"],title="PCA Pattern Visualization",template="plotly_white").show()
print("Variance explained by two PCA components:",round(pca.explained_variance_ratio_.sum()*100,2),"%")
# What-if sensitivity analysis
values=df["Monthly_Spending"].dropna(); q1,q3=values.quantile([.25,.75]); upper=q3+1.5*(q3-q1); filtered=values[values<=upper]
what_if=pd.DataFrame({"Scenario":["Original","Exclude values above IQR upper bound"],"Mean_Spending":[values.mean(),filtered.mean()],"Median_Spending":[values.median(),filtered.median()],"Records":[len(values),len(filtered)]}); what_if[["Mean_Spending","Median_Spending"]]=what_if[["Mean_Spending","Median_Spending"]].round(2)
print("WHAT-IF ANALYSIS"); display(what_if); px.bar(what_if,x="Scenario",y=["Mean_Spending","Median_Spending"],barmode="group",title="Effect of Extreme Values on Statistics",template="plotly_white").show()
# Statistical test
groups=[g["Monthly_Spending"].dropna().values for _,g in df.groupby("Region",observed=True)]; statistic,p_value=kruskal(*groups)
print("KRUSKAL-WALLIS: statistic=",round(statistic,4),"p-value=",round(p_value,6)); print("A post-hoc test is needed to determine which specific regions differ.")
# Automated report
highest_missing=profile["Missing_Percentage"].idxmax(); highest_spending_region=region_summary["Average_Spending"].idxmax(); corr=df[["Monthly_Income","Monthly_Spending"]].corr().iloc[0,1]
report=pd.DataFrame({"Metric":["Total records","Total features","Duplicate rows","Missing cells","Feature with highest missing percentage","Potential ML anomalies","Region with highest average spending","Income-spending correlation","Original mean spending","Mean after IQR sensitivity analysis","Kruskal-Wallis p-value"],"Result":[len(df),len(df.columns),int(df.duplicated().sum()),int(df.isna().sum().sum()),highest_missing,int((pred==-1).sum()),highest_spending_region,round(corr,4),round(values.mean(),2),round(filtered.mean(),2),round(p_value,6)]})
print("AUTOMATED INVESTIGATION REPORT"); display(report)
# Export files
df.to_csv("reality_lens_investigated_dataset.csv",index=False); profile.to_csv("reality_lens_data_profile.csv"); outlier_table.to_csv("reality_lens_outlier_analysis.csv",index=False); what_if.to_csv("reality_lens_what_if_analysis.csv",index=False); report.to_csv("reality_lens_investigation_report.csv",index=False)
print("PROJECT COMPLETE. Created 5 CSV reports in the current notebook folder.")
display(HTML("<div style='padding:20px;border:1px solid #94a3b8;border-radius:12px'><h2>REALITY LENS AI — Investigation Complete</h2><p>Data quality, distributions, correlations, outliers, anomaly detection, regional comparisons, PCA and what-if analysis are complete.</p><b>Note:</b> Synthetic demonstration data only. Correlation and anomaly flags do not establish causation or fraud.</div>"))
